<a href="https://colab.research.google.com/github/kimjiwoo2/Pill-agent/blob/develop/notebooks/yoonsoo/ys_paddle_OCR_baseline_0601.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pill Imprint OCR — PaddleOCR Baseline

**흐름:**
1. 환경 설치 (PaddlePaddle 우선 설치)
2. 경로 설정
3. Manifest 로드 및 target 레이블 정의
4. 이미지 경로 resolve
5. 공통 전처리
6. BBox crop 및 2차 OCR 전처리
7. PaddleOCR 추론
8. Metrics 평가
9. 실패 케이스 분석
10. DB 매핑 scaffold
11. PARSeq fine-tuning 데이터 추출
12. Synthetic data scaffold


## 0. Colab setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Paddle 패키지 설치
!pip uninstall -y torch torchvision torchaudio modelscope
!pip install paddlepaddle-gpu==3.1.0 -i \
    https://www.paddlepaddle.org.cn/packages/stable/cu118/
!pip install "paddleocr>=3.0.0"


Found existing installation: torch 2.11.0+cpu
Uninstalling torch-2.11.0+cpu:
  Successfully uninstalled torch-2.11.0+cpu
Found existing installation: torchvision 0.26.0+cpu
Uninstalling torchvision-0.26.0+cpu:
  Successfully uninstalled torchvision-0.26.0+cpu
Found existing installation: torchaudio 2.11.0+cpu
Uninstalling torchaudio-2.11.0+cpu:
  Successfully uninstalled torchaudio-2.11.0+cpu
Looking in indexes: https://www.paddlepaddle.org.cn/packages/stable/cu118/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 GB 915.2 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 13.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 14.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 699.9/699.9 MB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [3]:
!pip install rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 29.9 MB/s eta 0:00:00


In [4]:
from __future__ import annotations

import re
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import cv2
import numpy as np
import pandas as pd


from rapidfuzz.distance import Levenshtein
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

## 1. Paths

In [5]:
ZIP_PATH = "/content/drive/MyDrive/Pillot/dataset/pilliot_15k_optimized_leakage_free_split.zip"
!unzip -q "{ZIP_PATH}" -d /content

In [6]:
DATA_ROOT = Path("/content/pilliot_15k_optimized_leakage_free_split")

MANIFEST_DIR = DATA_ROOT / "manifests"
IMAGE_ROOT   = DATA_ROOT / "images"

WORK_DIR   = Path("/content/pillot_ocr_work")
CROP_DIR   = WORK_DIR / "crops"
RESULT_DIR = WORK_DIR / "results"

for d in [WORK_DIR, CROP_DIR, RESULT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

## 2. Manifest loading & target 레이블 정의

In [7]:
IGNORE_IMPRINT_TOKENS = {
    "", "NAN", "NONE", "NULL", "마크", "분할선", "없음", "무", "-",
}


def normalize_imprint(text: object, keep_separator: bool = False) -> str:
    """각인 라벨/예측값을 비교 가능한 형태로 정규화.
    - NaN, 비문자 토큰 → ''
    - 대문자 변환, 공백 제거, 허용 문자 외 제거
    """
    if pd.isna(text):
        return ""
    text = str(text).strip()
    if text.upper() in IGNORE_IMPRINT_TOKENS:
        return ""
    text = text.upper()
    text = re.sub(r"\s+", "", text)
    text = text.replace("분할선", "").replace("마크", "")
    allowed = r"[^0-9A-Z가-힣+\-/|]" if keep_separator else r"[^0-9A-Z가-힣+\-/]"
    text = re.sub(allowed, "", text)
    if text.upper() in IGNORE_IMPRINT_TOKENS:
        return ""
    return text


def normalize_prediction(text: object) -> str:
    return normalize_imprint(text, keep_separator=True)


def build_target_for_row(row: pd.Series) -> dict:
    """drug_dir 기반으로 앞/뒷면에 맞는 정답 레이블을 생성.
    - '앞면' → print_front만 정답
    - '뒷면' → print_back만 정답
    - 미확인  → front/back 둘 다 후보
    """
    front = normalize_imprint(row.get("print_front", ""))
    back  = normalize_imprint(row.get("print_back",  ""))

    drug_dir = str(row.get("drug_dir", "")).strip()
    if drug_dir == "앞면" and front:
        return {"target_text_front": front, "target_text_back": back,
                "target_candidates": [front], "target_text": front}
    if drug_dir == "뒷면" and back:
        return {"target_text_front": front, "target_text_back": back,
                "target_candidates": [back], "target_text": back}

    candidates = list(dict.fromkeys(t for t in [front, back] if t))
    return {"target_text_front": front, "target_text_back": back,
            "target_candidates": candidates, "target_text": "/".join(candidates)}


def load_manifest(data_root: Path,
                  manifest_name: str = "bbox_manifest_val_with_attributes.csv") -> pd.DataFrame:
    path = data_root / "manifests" / manifest_name
    if not path.exists():
        raise FileNotFoundError(f"Manifest not found: {path}")
    df = pd.read_csv(path, low_memory=False)
    required = {"dataset_type", "split_type", "image_file"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns: {sorted(missing)}")
    return df


def filter_ocr_candidates(
    df: pd.DataFrame,
    split: Optional[str] = None,
    dataset_type: Optional[str] = "single",
    limit: Optional[int] = None,
    seed: int = 42,
) -> pd.DataFrame:
    """OCR 평가 후보 필터링.
    - has_print=True: 각인 있는 알약만
    - dataset_type='single': combination 제외
    - limit: item_seq 단위 샘플링
    """
    out = df.copy()

    if "for_ocr_eval_candidate" in out.columns:
        out = out[out["for_ocr_eval_candidate"].astype(str).str.lower().eq("true")]
    if "has_print" in out.columns:
        out = out[out["has_print"].astype(str).str.lower().eq("true")]
    if split:
        out = out[out["split_type"].eq(split)]
    if dataset_type:
        out = out[out["dataset_type"].eq(dataset_type)]

    if out.empty:
        return pd.DataFrame(columns=list(df.columns) +
                            ["target_text_front", "target_text_back",
                             "target_candidates", "target_text"])

    target_cols = out.apply(build_target_for_row, axis=1, result_type="expand")
    out = pd.concat([out, target_cols], axis=1)
    out = out[out["target_text"].ne("")]

    if limit:
        import random
        random.seed(seed)
        groups = out.groupby("item_seq")
        all_kinds = list(groups.groups.keys())
        random.shuffle(all_kinds)
        selected, total = [], 0
        for kind in all_kinds:
            count = len(groups.get_group(kind))
            if total + count > limit:
                continue
            selected.append(kind)
            total += count
            if total >= limit * 0.9:
                break
        out = out[out["item_seq"].isin(selected)]

    return out.reset_index(drop=True)


df_val  = load_manifest(DATA_ROOT)
df_eval = filter_ocr_candidates(df_val, dataset_type="single", limit=500)
print(f"val manifest: {df_val.shape}, 평가셋: {df_eval.shape}")
display(df_eval[["image_file", "target_text_front", "target_text_back", "target_text"]])

val manifest: (3784, 40), 평가셋: (454, 44)


,image_file,target_text_front,target_text_back,target_text
0,K-012757_0_2_0_0_75_320_200.png,JW3,,JW3
1,K-012757_0_2_0_1_75_000_200.png,JW3,,JW3
2,K-012757_0_2_0_1_75_260_200.png,JW3,,JW3
3,K-012757_0_2_0_2_75_220_200.png,JW3,,JW3
4,K-012757_0_2_0_2_75_280_200.png,JW3,,JW3
...,...,...,...,...
449,K-035811_0_0_1_0_90_100_200.png,UK,OC375,UK/OC375
450,K-035811_0_0_1_0_90_220_200.png,UK,OC375,UK/OC375
451,K-035811_0_0_1_0_90_300_200.png,UK,OC375,UK/OC375
452,K-035811_0_0_1_1_90_060_200.png,UK,OC375,UK/OC375


## 3. Image path resolving

In [8]:
def _zip_stem(name: str) -> str:
    return Path(str(name)).stem


def resolve_image_path(row: pd.Series, data_root: Path = DATA_ROOT) -> Path:
    """split 컬럼 우선 사용, 없으면 split_type 폴백."""
    split_col      = row.get("split") or row.get("split_type", "val")
    dataset_type   = str(row["dataset_type"])
    image_zip_stem = _zip_stem(row["image_zip_name"])
    image_file     = str(row["image_file"])
    return data_root / "images" / str(split_col) / dataset_type / image_zip_stem / image_file


def report_missing_images(df: pd.DataFrame, n: int = 10) -> None:
    paths = df.apply(resolve_image_path, axis=1)
    missing = [not p.exists() for p in paths]
    print(f"이미지 확인: {sum(not m for m in missing)} 존재 / {sum(missing)} 누락 (총 {len(df)})")
    if any(missing):
        print("누락 zip 목록:")
        print(df[missing]["image_zip_name"].value_counts().head(n))


report_missing_images(df_eval)

이미지 확인: 454 존재 / 0 누락 (총 454)


## 4. 공통 전처리

In [9]:
@dataclass
class CommonPreprocessConfig:
    apply_awb: bool = True
    denoise_method: str = "bilateral"  # "none" | "bilateral" | "nlm"
    bilateral_d: int = 5
    bilateral_sigma_color: int = 35
    bilateral_sigma_space: int = 35
    nlm_h: int = 3
    target_size: int = 640
    pad_color: tuple[int, int, int] = (114, 114, 114)
    normalize_pixels: bool = False


def read_bgr(path: Path) -> np.ndarray:
    path = Path(path)
    if path.suffix.lower() in {".heic", ".heif"}:
        try:
            import pillow_heif
            from PIL import Image
            pillow_heif.register_heif_opener()
            rgb = np.array(Image.open(path).convert("RGB"))
            return cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
        except ImportError as exc:
            raise ImportError("pillow-heif 필요: pip install pillow-heif") from exc
    image = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if image is None:
        raise FileNotFoundError(path)
    if image.ndim == 2:
        return cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
    if image.shape[2] == 4:
        return cv2.cvtColor(image, cv2.COLOR_BGRA2BGR)
    return image


def gray_world_awb_bgr(image_bgr: np.ndarray) -> np.ndarray:
    image = image_bgr.astype(np.float32)
    means = image.reshape(-1, 3).mean(axis=0)
    gray_mean = means.mean()
    scale = gray_mean / np.maximum(means, 1e-6)
    return np.clip(image * scale, 0, 255).astype(np.uint8)


def denoise_bgr(image_bgr: np.ndarray, cfg: CommonPreprocessConfig) -> np.ndarray:
    if cfg.denoise_method == "none":
        return image_bgr
    if cfg.denoise_method == "bilateral":
        return cv2.bilateralFilter(image_bgr, cfg.bilateral_d,
                                   cfg.bilateral_sigma_color, cfg.bilateral_sigma_space)
    if cfg.denoise_method == "nlm":
        return cv2.fastNlMeansDenoisingColored(image_bgr, None, cfg.nlm_h, cfg.nlm_h, 7, 21)
    raise ValueError(f"Unknown denoise_method: {cfg.denoise_method}")


def common_preprocess_bgr(
    image_bgr: np.ndarray,
    cfg: CommonPreprocessConfig = CommonPreprocessConfig(),
) -> np.ndarray:
    out = image_bgr
    if cfg.apply_awb:
        out = gray_world_awb_bgr(out)
    return denoise_bgr(out, cfg)

## 5. BBox crop 및 2차 OCR 전처리

In [10]:
@dataclass
class CropConfig:
    margin_ratio: float = 0.16
    min_size: int = 96
    target_size: int = 384
    use_bbox: bool = True
    apply_common_preprocess: bool = True


def crop_with_bbox(image_bgr: np.ndarray, row: pd.Series, cfg: CropConfig) -> np.ndarray:
    h, w = image_bgr.shape[:2]
    if not cfg.use_bbox or any(pd.isna(row.get(k)) for k in ["bbox_x","bbox_y","bbox_w","bbox_h"]):
        return image_bgr
    x, y, bw, bh = [float(row[k]) for k in ["bbox_x","bbox_y","bbox_w","bbox_h"]]
    margin = cfg.margin_ratio * max(bw, bh)
    x1 = max(0, int(round(x - margin)))
    y1 = max(0, int(round(y - margin)))
    x2 = min(w, int(round(x + bw + margin)))
    y2 = min(h, int(round(y + bh + margin)))
    crop = image_bgr[y1:y2, x1:x2]
    return image_bgr if min(crop.shape[:2]) < cfg.min_size else crop


def enhance_crop_for_ocr_bgr(image_bgr: np.ndarray, target_size: int = 384) -> np.ndarray:
    h, w = image_bgr.shape[:2]
    scale = target_size / max(h, w)
    if scale != 1:
        image_bgr = cv2.resize(image_bgr, (int(w*scale), int(h*scale)),
                               interpolation=cv2.INTER_CUBIC)
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    return cv2.cvtColor(clahe.apply(gray), cv2.COLOR_GRAY2BGR)


def make_crop(
    row: pd.Series,
    crop_cfg: CropConfig = CropConfig(),
    preprocess_cfg: CommonPreprocessConfig = CommonPreprocessConfig(),
) -> np.ndarray:
    path = resolve_image_path(row)
    image_bgr = read_bgr(path)
    if crop_cfg.apply_common_preprocess:
        image_bgr = common_preprocess_bgr(image_bgr, preprocess_cfg)
    return enhance_crop_for_ocr_bgr(crop_with_bbox(image_bgr, row, crop_cfg), crop_cfg.target_size)


def save_eval_crops(df: pd.DataFrame, out_dir: Path = CROP_DIR, limit: Optional[int] = None) -> pd.DataFrame:
    rows = df.head(limit).copy() if limit else df.copy()
    records, skip_reasons = [], []

    for idx, row in tqdm(rows.iterrows(), total=len(rows), desc="cropping"):
        try:
            crop_bgr = make_crop(row)
            safe_name = str(row["image_file"]).replace("/", "_").replace("\\", "_")
            out_path = out_dir / f"{idx:06d}_{safe_name}"
            cv2.imwrite(str(out_path), crop_bgr)
            record = row.to_dict()
            record["crop_path"] = str(out_path)
            records.append(record)
        except Exception as exc:
            skip_reasons.append({"image_file": row.get("image_file"), "error": str(exc)})

    if skip_reasons:
        skip_df = pd.DataFrame(skip_reasons)
        print(f"\nskip {len(skip_df)}건 — 오류 유형별 요약:")
        print(skip_df["error"].value_counts().head(10))

    return pd.DataFrame(records)


df_crops = save_eval_crops(df_eval)
print(f"crop 완료: {len(df_crops)}건")
display(df_crops[["image_file", "target_text_front", "target_text_back", "target_text", "crop_path"]].head())

cropping:   0%|          | 0/454 [00:00<?, ?it/s]

crop 완료: 454건


,image_file,target_text_front,target_text_back,target_text,crop_path
0,K-012757_0_2_0_0_75_320_200.png,JW3,,JW3,/content/pillot_ocr_work/crops/000000_K-012757...
1,K-012757_0_2_0_1_75_000_200.png,JW3,,JW3,/content/pillot_ocr_work/crops/000001_K-012757...
2,K-012757_0_2_0_1_75_260_200.png,JW3,,JW3,/content/pillot_ocr_work/crops/000002_K-012757...
3,K-012757_0_2_0_2_75_220_200.png,JW3,,JW3,/content/pillot_ocr_work/crops/000003_K-012757...
4,K-012757_0_2_0_2_75_280_200.png,JW3,,JW3,/content/pillot_ocr_work/crops/000004_K-012757...


## 6. PaddleOCR

In [11]:
# 런타임 재시작 후 버전 확인
import paddle
print(paddle.__version__)   # 3.1.0 이어야 함

Error: Can not import paddle core while this file exists: /usr/local/lib/python3.12/dist-packages/paddle/base/libpaddle.so


ImportError: libcuda.so.1: cannot open shared object file: No such file or directory

In [12]:
# PaddleOCR 3.x는 재초기화 시 리소스 충돌 가능성 있음.
# 같은 런타임 내에서는 모델 객체를 한 번만 생성하고 재사용.
_PADDLE_OCR_MODEL = None


def get_paddleocr_model():
    from paddleocr import PaddleOCR
    global _PADDLE_OCR_MODEL
    if _PADDLE_OCR_MODEL is not None:
        return _PADDLE_OCR_MODEL
    _PADDLE_OCR_MODEL = PaddleOCR(
        lang="korean",
        use_textline_orientation=True,   # 2.x use_angle_cls → 3.x 변경
        use_doc_orientation_classify=False,  # 알약 단순 이미지에 불필요
        use_doc_unwarping=False,
        det_db_box_thresh=0.25,
        det_db_unclip_ratio=1.8,
    )
    return _PADDLE_OCR_MODEL


def run_paddleocr(df_crops: pd.DataFrame) -> pd.DataFrame:
    """PaddleOCR 3.x 추론.
    - 2.x: ocr.ocr(path, cls=True)  → output[0] = [[[bbox], (text, conf)], ...]
    - 3.x: ocr.predict(path)         → output = [PredResult, ...]
           PredResult.rec_texts / PredResult.rec_scores 로 접근
    """
    ocr_model = get_paddleocr_model()
    results = []

    for _, row in tqdm(df_crops.iterrows(), total=len(df_crops), desc="paddleocr"):
        output = ocr_model.predict(row["crop_path"])
        pieces, confs = [], []

        for pred in (output or []):
            for text, conf in zip(pred.rec_texts, pred.rec_scores):
                text = normalize_prediction(text)
                if text:
                    pieces.append(text)
                    confs.append(float(conf))

        record = row.to_dict()
        record["pred_text"] = "".join(pieces)
        record["ocr_conf"]  = float(np.mean(confs)) if confs else 0.0
        record["raw_ocr"]   = repr(output)
        results.append(record)

    out = pd.DataFrame(results)
    out.to_csv(RESULT_DIR / "paddleocr_results.csv", index=False, encoding="utf-8-sig")
    return out


paddle_results = run_paddleocr(df_crops)
display(paddle_results[["image_file", "target_text", "pred_text", "ocr_conf"]].head(20))

ImportError: /usr/local/lib/python3.12/dist-packages/torch/lib/libtorch_cuda.so: undefined symbol: ncclCommShrink

## 7. Metrics

In [ ]:
def char_accuracy(target: str, pred: str) -> float:
    """문자 단위 유사도. 1.0이면 완전 일치."""
    target = normalize_prediction(target)
    pred   = normalize_prediction(pred)
    if not target:
        return float(pred == "")
    return max(0.0, 1.0 - Levenshtein.distance(target, pred) / len(target))


def parse_target_candidates(value: object) -> list[str]:
    """target_candidates 또는 target_text 문자열을 후보 리스트로 변환."""
    if isinstance(value, list):
        return [normalize_prediction(v) for v in value if normalize_prediction(v)]
    if pd.isna(value):
        return []
    return [x for x in [normalize_prediction(p) for p in str(value).split("/")] if x]


def best_candidate_score(target_candidates: object, pred: str) -> tuple[str, bool, float]:
    """후보 중 OCR 결과와 가장 가까운 후보를 반환."""
    candidates = parse_target_candidates(target_candidates)
    pred = normalize_prediction(pred)
    if not candidates:
        return "", pred == "", float(pred == "")
    scored = [(t, t == pred, char_accuracy(t, pred)) for t in candidates]
    return max(scored, key=lambda x: (x[1], x[2]))


def evaluate_ocr_results(df: pd.DataFrame, label: str = "") -> dict:
    """Exact Match와 Char Accuracy를 계산하고 출력."""
    exact, chars = [], []
    for _, row in df.iterrows():
        pred = normalize_prediction(row["pred_text"])
        _, is_exact, score = best_candidate_score(
            row.get("target_candidates", row["target_text"]), pred
        )
        exact.append(is_exact)
        chars.append(score)

    result = {
        "n": len(df),
        "exact_match":   round(float(np.mean(exact)), 4) if exact else 0.0,
        "char_accuracy": round(float(np.mean(chars)), 4) if chars else 0.0,
    }
    prefix = f"[{label}] " if label else ""
    print(f"{prefix}n={result['n']}  Exact Match={result['exact_match']:.3f}  "
          f"Char Accuracy={result['char_accuracy']:.3f}")
    return result


evaluate_ocr_results(paddle_results, label="PaddleOCR")

## 8. 실패 케이스 분석

In [ ]:
def add_error_columns(df: pd.DataFrame) -> pd.DataFrame:
    """OCR 결과표에 평가 컬럼 추가."""
    out = df.copy()
    out["pred_norm"] = out["pred_text"].map(normalize_prediction)
    best = [
        best_candidate_score(t, p)
        for t, p in zip(out.get("target_candidates", out["target_text"]), out["pred_norm"])
    ]
    out["matched_target"] = [x[0] for x in best]
    out["exact"]          = [x[1] for x in best]
    out["char_acc"]       = [x[2] for x in best]
    return out


def show_failures(results: pd.DataFrame, n: int = 30) -> pd.DataFrame:
    """정답 못 맞힌 케이스를 char_acc 오름차순으로 출력."""
    err = add_error_columns(results)
    cols = ["image_file", "target_text", "matched_target", "pred_text",
            "ocr_conf", "char_acc", "crop_path"]
    return err.sort_values(["exact", "char_acc", "ocr_conf"],
                           ascending=[True, True, True])[cols].head(n)


display(show_failures(paddle_results, n=30))

## 9. DB 매핑 scaffold

In [ ]:
# OCR에서 자주 혼동되는 문자 그룹
OCR_CONFUSION_GROUPS = [
    ("0", "O"),
    ("1", "I", "L"),
    ("8", "B"),
]


def generate_ocr_text_variants(text: str, max_variants: int = 32) -> list[str]:
    """혼동 문자 치환 후보를 생성. DB 후보 비교 시 보조 후보로만 사용."""
    base = normalize_prediction(text)
    variants = {base}
    for group in OCR_CONFUSION_GROUPS:
        next_variants = set(variants)
        for value in variants:
            for src in group:
                for dst in group:
                    if src != dst and src in value:
                        next_variants.add(value.replace(src, dst))
                        if len(next_variants) >= max_variants:
                            return sorted(next_variants)
        variants = next_variants
    return sorted(variants)


def text_similarity_with_variants(pred_text: str, target_text: str) -> float:
    """OCR 보정 후보 중 가장 가까운 유사도 반환."""
    target = normalize_prediction(target_text)
    if not target:
        return 0.0
    return max(char_accuracy(target, v) for v in generate_ocr_text_variants(pred_text))


def weighted_candidate_score(
    pred_text: str,
    candidate_row: pd.Series,
    pred_shape: Optional[str] = None,
    pred_color: Optional[str] = None,
    weights: dict | None = None,
) -> float:
    """OCR + 속성 분류 결과를 합산해 DB 후보 점수 계산.
    pred_shape / pred_color는 추후 속성 분류기 결과 연결 자리.
    """
    weights = weights or {"ocr": 0.70, "shape": 0.15, "color": 0.15}

    front = normalize_imprint(candidate_row.get("print_front", ""))
    back  = normalize_imprint(candidate_row.get("print_back",  ""))
    imprint_score = max(
        [text_similarity_with_variants(pred_text, t) for t in [front, back] if t] or [0.0]
    )

    shape_score = float(str(candidate_row.get("drug_shape", "")) == str(pred_shape)) if pred_shape else 0.0
    color_score = 0.0
    if pred_color:
        color_candidates = {str(candidate_row.get("color_class1", "")),
                            str(candidate_row.get("color_class2", ""))}
        color_score = float(str(pred_color) in color_candidates)

    return (weights["ocr"] * imprint_score
            + weights["shape"] * shape_score
            + weights["color"] * color_score)


def search_db_candidates(
    pred_text: str,
    db_df: pd.DataFrame,
    pred_shape: Optional[str] = None,
    pred_color: Optional[str] = None,
    top_k: int = 10,
) -> pd.DataFrame:
    """OCR + 속성 결과로 DB 후보 Top-K 추출.
    현재는 manifest를 DB로 사용. 추후 HIRA/의약품 DB로 교체 가능.
    """
    scored = db_df.copy()
    scored["candidate_score"] = scored.apply(
        lambda row: weighted_candidate_score(pred_text, row, pred_shape, pred_color), axis=1
    )
    cols = [c for c in ["candidate_score", "dl_name", "item_seq",
                         "print_front", "print_back", "drug_shape",
                         "color_class1", "color_class2"] if c in scored.columns]
    return scored.sort_values("candidate_score", ascending=False)[cols].head(top_k)


# 사용 예시:
# search_db_candidates("TYLENOL", df_val, pred_shape="장방형", pred_color="하양", top_k=5)

## 10. PARSeq fine-tuning 데이터 추출

In [ ]:
def export_recognition_training_csv(
    df_crops: pd.DataFrame,
    out_path: Path = RESULT_DIR / "parseq_recognition_train.csv",
) -> Path:
    """PARSeq 학습용 CSV 생성.
    앞/뒤 후보가 둘 다 있는 이미지는 어느 면인지 확정할 수 없으므로 제외.
    """
    rec = df_crops[["crop_path", "target_text", "target_candidates"]].copy()
    rec["target_candidates"] = rec["target_candidates"].map(parse_target_candidates)
    rec = rec[rec["target_candidates"].map(len).eq(1)].copy()
    rec["target_text"] = rec["target_candidates"].map(lambda xs: xs[0])
    rec["target_text"] = rec["target_text"].map(normalize_prediction)
    rec = rec[rec["target_text"].ne("")]
    rec[["crop_path", "target_text"]].to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"저장 완료: {out_path}  ({len(rec)}건)")
    return out_path


export_recognition_training_csv(df_crops)